# Проект\. Часть 7

## HR Analytics: Employee Attrition & Performance

Источник \(данные не менялись\): Kaggle https://www\.kaggle\.com/datasets/pavansubhasht/ibm\-hr\-analytics\-attrition\-dataset/data

Цель проекта: исследовать факторы, влияющие на текучесть кадров \(Attrition\) и определить, какие характеристики сотрудников повышают вероятность их ухода из компании\.

Задача для предсказания: предсказать, уйдёт ли сотрудник из компании, основываясь на данных о нём и его работе \(задача классификации\)\. Эта задача напрямую связана с основной целью проекта — анализом факторов текучести кадров\.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [2]:
# загрузка данных
data = pd.read_csv("datasets/HR-Employee-Attrition.csv")

columns_to_drop = ['EducationField', 'JobRole', 'PerformanceRating', 'EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours', 'HourlyRate', 'MonthlyRate']
data = data.drop(columns=columns_to_drop)

data = pd.get_dummies(data, columns=['Attrition', 'BusinessTravel', 'Department', 'Gender', 'OverTime', 'MaritalStatus'], drop_first=True)

Формирование признаков и целевой переменной:

Целевая переменная — увольнение сотрудника \(1 = ушёл, 0 = остался\)\.

In [3]:
y = data["Attrition_Yes"]

X = data.drop(columns="Attrition_Yes")

print("Количество признаков:", X.shape[1])

Количество признаков: 28


Разделение на обучающую и тестовую выборки:

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=1/3,
    stratify=y,
    random_state=0
)

Параметр stratify=y сохраняет одинаковую долю уволившихся сотрудников в обеих выборках\.

### Построение нейронной сети \(MLPClassifier\):

In [5]:
# Создаём простую нейронную сеть
nn_model = MLPClassifier(
    hidden_layer_sizes=(20,20),  # два скрытых слоя
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=1
)

# Обучаем модель
nn_model.fit(X_train, y_train)

MLPClassifier(hidden_layer_sizes=(20, 20), max_iter=300, random_state=1)

Используется многослойный перцептрон \(MLP\)\. Архитектура сети включает два скрытых слоя по 20 нейронов\.

Предсказание модели:

In [6]:
# Предсказанные классы
y_pred = nn_model.predict(X_test)

# Предсказанные вероятности
y_pred_proba = nn_model.predict_proba(X_test)[:,1]

### Оценка качества модели:

Для оценки качества используется метрика accuracy — доля правильно классифицированных сотрудников\.

In [7]:
train_accuracy = nn_model.score(X_train, y_train)
test_accuracy = nn_model.score(X_test, y_test)

print("Accuracy на обучающей выборке:", train_accuracy)
print("Accuracy на тестовой выборке:", test_accuracy)

Accuracy на обучающей выборке: 0.5071428571428571
Accuracy на тестовой выборке: 0.5


Качество на обучающей и тестовой выборках близкое, это означает, что модель не сильно переобучилась, но точность около 0\.5 означает, что сеть фактически не научилась извлекать закономерности и работает почти как случайное угадывание\.

In [8]:
print("Количество итераций обучения:", nn_model.n_iter_)

Количество итераций обучения: 51


### Улучшение нейронной сети:

Попробуем улучшить модель за счёт масштабирования признаков и усложнения структуры сети \(больше слоёв и нейронов в них\)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Что изменяем: больше слоев, адаптивная скорость обучения, early stopping, больше итераций

In [10]:
new_nn_model = MLPClassifier(
    hidden_layer_sizes=(64,32,16),   # больше нейронов и слоев
    activation='relu',
    solver='adam',
    alpha=0.0005,            # небольшая регуляризация
    learning_rate='adaptive',
    max_iter=800,
    early_stopping=True,
    random_state=1
)

new_nn_model.fit(X_train_scaled, y_train)

MLPClassifier(alpha=0.0005, early_stopping=True,
              hidden_layer_sizes=(64, 32, 16), learning_rate='adaptive',
              max_iter=800, random_state=1)

Проверка качества:

In [11]:
train_accuracy = new_nn_model.score(X_train_scaled, y_train)
test_accuracy = new_nn_model.score(X_test_scaled, y_test)

print("Accuracy на обучающей выборке:", train_accuracy)
print("Accuracy на тестовой выборке:", test_accuracy)

print("Количество итераций обучения:", new_nn_model.n_iter_)

Accuracy на обучающей выборке: 0.8387755102040816
Accuracy на тестовой выборке: 0.8387755102040816
Количество итераций обучения: 12


Для улучшения качества модели была выполнена стандартизация признаков и изменена архитектура нейронной сети\. В новой конфигурации сеть содержит три скрытых слоя \(64, 32 и 16 нейронов\), использует функцию активации ReLU и алгоритм оптимизации Adam\. Дополнительно применена ранняя остановка обучения \(early stopping\), позволяющая избежать переобучения\. 

Вывод: После изменения структуры качество модели увеличилось и составило 0\.84 на обучающей и тестовой выборках, что существенно выше качества первоначальной модели и свидетельствует о том, что нейронная сеть хорошо выявляет закономерности, связанные с уходом сотрудников из компании, при этом без заметного переобучения\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=ff5c65e0-9edf-47b2-8d5a-439e7f5c1ece' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>